In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

In [2]:
ss_ds = xr.open_dataset('sg195_Shilshole_Shakedown_timeseries.nc')
display(ss_ds)

<xarray.Dataset> Size: 85kB
Dimensions:                                   (gps_info: 6, sg_data_point: 466,
                                               trajectory: 2, dive: 2)
Coordinates:
    ctd_time                                  (sg_data_point) datetime64[ns] 4kB ...
    ctd_depth                                 (sg_data_point) float32 2kB ...
    latitude                                  (sg_data_point) float32 2kB ...
    longitude                                 (sg_data_point) float32 2kB ...
  * trajectory                                (trajectory) int32 8B 1 2
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/66)
    gps_info_dive_number                      (gps_info) int32 24B ...
    sg_data_point_dive_number                 (sg_data_point) int32 2kB ...
    log_gps_time                              (gps_info) datetime64[ns] 48B ...
    time                                      (sg_data_point) datetime64[ns] 4kB ...
    pressure                                  (sg_data_point) float32 2kB ...
    depth                                     (sg_data_point) float32 2kB ...
    ...                                        ...
    end_longitude                             (dive) float32 8B ...
    depth_avg_curr_east                       (dive) float32 8B ...
    depth_avg_curr_north                      (dive) float32 8B ...
    depth_avg_curr_qc                         (dive) |S1 2B ...
    latlong_qc                                (dive) |S1 2B ...
    glider                                    |S12 12B ...
Attributes: (12/48)
    project:                         Shilshole Shakedown
    title:                           Physical, chemical, and biological data ...
    summary:                         SG195 Shilshole Shakedown
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-04-05T22:04:25Z
    uuid:                            e44ac894-f39e-11ee-8bfb-cb77ccca147b
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [3]:
print(os.getcwd())

C:\Users\lydia\Seagliders


In [4]:
#Align time with sg_data_point and apply offset (if needed)
print(ss_ds['time'][0].values)
adjusted_time = pd.to_datetime(ss_ds['time'].values) + pd.DateOffset(years=19, months=7, days=16)
ss_ds = ss_ds.assign_coords(time=('sg_data_point', adjusted_time))

print(ss_ds['time'][0].values)

2004-08-20T20:48:31.796999936
2024-04-05T20:48:31.796999936


In [5]:
ss_ds['PAR_470nm'] = ss_ds['eng_wlbb2fl_sig470nm']
ss_ds['particle_concentration_700nm'] = ss_ds['eng_wlbb2fl_sig700nm']
ss_ds['chlorophyll_695nm'] = ss_ds['eng_wlbb2fl_sig695nm']
ss_ds['dissolved_oxygen'] = ss_ds['aanderaa4330_dissolved_oxygen']

# add metadata
ss_ds['PAR_470nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig470nm'
ss_ds['particle_concentration_700nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig700nm'
ss_ds['chlorophyll_695nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig695nm'
ss_ds['dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_dissolved_oxygen'

ss_ds = ss_ds.assign_coords(time=('sg_data_point', adjusted_time))

#Select the relevant variables
new_ss_ds = ss_ds[['time', 'latitude', 'longitude','temperature', 'salinity', 'dissolved_oxygen', 'PAR_470nm', 'particle_concentration_700nm', 'chlorophyll_695nm']]

#Convert to DataFrame and save
new_ss_ds.to_dataframe().reset_index().to_csv('sg195_April_2024_Shilshole_Test_timeseries_cleaned.csv', index=False)
new_ss_ds.to_netcdf('sg195_April_2024_Shilshole_Test_timeseries_cleaned.nc')
display(ss_ds)

<xarray.Dataset> Size: 93kB
Dimensions:                                   (gps_info: 6, sg_data_point: 466,
                                               trajectory: 2, dive: 2)
Coordinates:
    time                                      (sg_data_point) datetime64[ns] 4kB ...
    ctd_time                                  (sg_data_point) datetime64[ns] 4kB ...
    ctd_depth                                 (sg_data_point) float32 2kB 0.4...
    latitude                                  (sg_data_point) float32 2kB 47....
    longitude                                 (sg_data_point) float32 2kB -12...
  * trajectory                                (trajectory) int32 8B 1 2
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/69)
    gps_info_dive_number                      (gps_info) int32 24B ...
    sg_data_point_dive_number                 (sg_data_point) int32 2kB ...
    log_gps_time                              (gps_info) datetime64[ns] 48B ...
    pressure                                  (sg_data_point) float32 2kB ...
    depth                                     (sg_data_point) float32 2kB ...
    speed_gsm                                 (sg_data_point) float32 2kB ...
    ...                                        ...
    latlong_qc                                (dive) |S1 2B ...
    glider                                    |S12 12B ...
    PAR_470nm                                 (sg_data_point) float32 2kB 177...
    particle_concentration_700nm              (sg_data_point) float32 2kB 1.0...
    chlorophyll_695nm                         (sg_data_point) float32 2kB 1.8...
    dissolved_oxygen                          (sg_data_point) float32 2kB nan...
Attributes: (12/48)
    project:                         Shilshole Shakedown
    title:                           Physical, chemical, and biological data ...
    summary:                         SG195 Shilshole Shakedown
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-04-05T22:04:25Z
    uuid:                            e44ac894-f39e-11ee-8bfb-cb77ccca147b
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [7]:
ss_ds = xr.open_dataset('sg000_Shilshole_Test_2024.04.05_timeseries.nc')

#Apply time apply offset (if needed)
#Apply time apply offset (if needed)
adjusted_start = pd.to_datetime(ss_ds['start_time'].values) + pd.DateOffset(years=19, months=7, days=16)
adjusted_end = pd.to_datetime(ss_ds['end_time'].values) + pd.DateOffset(years=19, months=7, days=16)
shilshole_ds = ss_ds.assign_coords(start_time=('dive', adjusted_start), end_time=('dive', adjusted_end))

ss_ds['U_DAC'] = ss_ds['depth_avg_curr_east']
ss_ds['V_DAC'] = ss_ds['depth_avg_curr_north']

# add metadata
ss_ds['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
ss_ds['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_ss_ds = ss_ds[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_ss_ds)

#Convert to DataFrame and save
new_ss_ds.to_dataframe().reset_index().to_csv('sg195_April_2024_Shilshole_Test_timeseries_DAC_cleaned.csv', index=False)
new_ss_ds.to_netcdf('sg195_April_2024_Shilshole_Test_DAC_timeseries_cleaned.nc')


<xarray.Dataset> Size: 240B
Dimensions:          (dive: 6)
Dimensions without coordinates: dive
Data variables:
    U_DAC            (dive) float32 24B ...
    V_DAC            (dive) float32 24B ...
    start_time       (dive) datetime64[ns] 48B 2004-08-20T17:24:50 ... 2004-0...
    end_time         (dive) datetime64[ns] 48B 2004-08-20T17:45:29 ... 2004-0...
    start_latitude   (dive) float32 24B ...
    end_latitude     (dive) float32 24B ...
    start_longitude  (dive) float32 24B ...
    end_longitude    (dive) float32 24B ...
Attributes: (12/48)
    project:                         Shilshole Test 2024.04.05
    title:                           Physical, chemical, and biological data ...
    summary:                         SG000 Shilshole Test 2024.04.05
    source:                          Seaglider SG000
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-04-05T23:04:49Z
    uuid:                            e36f7920-f3a7-11ee-8bfb-cb77ccca147b
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6